In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

## Question 1
Which is closest to the probability that this t-test will be able to detect the event at event_time + 1 with the given code? 

- A: 0.1

- B: 0.7

- C: 0.5

- D: 0.3

In [2]:
num = 1000
event_time = num // 2
n_runs = 1000
detected = 0

rng = np.random.default_rng(42)
days = np.arange(num)

for _ in range(n_runs):
    R_market = rng.normal(0, 1, num) + days / num
    R_target = ( 2 + R_market + rng.normal(0, 1, num)+ 2 * (days == event_time + 1))

    X = sm.add_constant(R_market)
    results = sm.OLS(R_target[:event_time], X[:event_time]).fit()
    resid = R_target - results.predict(X)

    t_stat = resid[event_time + 1] / resid[:event_time].std(ddof=2)

    if t_stat > 1.96 or t_stat < -1.96:
        detected += 1

print(f"Detected: {detected}/{n_runs}")
print(f"Estimated probability: {detected / n_runs:.3f}")

Detected: 505/1000
Estimated probability: 0.505


Answer:  C   
_____

## Question 2
Use the same code but put np.random.seed(0) at the beginning of each loop to ensure that you are performing placebo tests on a fixed dataset.   
   
Perform a placebo test by setting the fictitious event_time to all possible times, while leaving the event in R_target at just the 1 time.   

The placebo test trains itself on the data leading up to the fictitious event. About what fraction of placebo tests seem to detect an event at the fictitious event time?   


- A: 0.10

- B: 0.05

- C: 0.02

- D: 0.15

In [3]:
num = 1000
days = np.arange(num)
real_event_day = num // 2 + 1  # 501

detected = 0
n_placebos = 0

for placebo_day in range(3, num):
    np.random.seed(0)

    if placebo_day == real_event_day:
        continue

    R_market = np.random.normal(0, 1, num) + days / num
    R_target = (2 + R_market + np.random.normal(0, 1, num)+ 2 * (days == real_event_day))

    X = sm.add_constant(R_market)
    results = sm.OLS(R_target[:placebo_day], X[:placebo_day]).fit()
    resid = R_target - results.predict(X)

    t_stat = resid[placebo_day] / resid[:placebo_day].std(ddof=2)
    if abs(t_stat) > 1.96:
        detected += 1
    n_placebos += 1

print(f"{detected}/{n_placebos} = {detected / n_placebos:.3f}")
# 48/996 = 0.048

48/996 = 0.048


Answer:    B 
_____

## Question 3
Do the same placebo test, but this time only run the test 20 times before and twenty times after the actual event.   

On average (over many runs of the code), what fraction of the 40 placebo tests get a higher t-value than the actual event?   

This time, adjust np.random.seed() to represent a different dataset when needed. 
   

- A: 0.35

- B: 0.05

- C: 0.15

- D: 0.25

In [4]:
def make_error(corr_const, num): 
 
    sigma = 5 * 1 / np.sqrt((1 - corr_const)**2 / (1 - corr_const**2)) 
    err = list() 
    prev = np.random.normal(0, sigma) 

    for n in range(num):  
    
        prev = corr_const * prev + (1 - corr_const) * np.random.normal(0, sigma) 
        err.append(prev)
         
    return np.array(err) 

In [5]:
def get_t_value(R_target, R_market, test_event_time):
    model = sm.OLS(R_target[:test_event_time], sm.add_constant(R_market[:test_event_time])).fit()
    resid = R_target - model.predict(sm.add_constant(R_market))
    return resid[test_event_time + 1] / resid[:test_event_time].std(ddof=2)

num = 1000
real_event_time = num // 2
fractions = []

for seed in range(1000):
    np.random.seed(seed)
    days = np.arange(num)
    R_market = np.random.normal(0, 1, num) + days / num
    R_target = 2 + R_market + np.random.normal(0, 1, num) + (days == real_event_time + 1) * 2

    actual_t = get_t_value(R_target, R_market, real_event_time)
    placebo_times = list(range(real_event_time - 20, real_event_time)) + list(range(real_event_time + 1, real_event_time + 21))
    placebo_t_values = [get_t_value(R_target, R_market, placebo_time) for placebo_time in placebo_times]

    fractions.append(np.mean(np.abs(placebo_t_values) > abs(actual_t)))

print(np.mean(fractions))

0.1461


Answer:   C
_____

## Question 4
Do the same thing as in question 2, but this time use make_error with corr_const = 0.9 to generate the error for R_target instead of np.random.normal.   

Consider before attempting this: Would you expect this kind of dataset, where errors are not independent, to result in more or fewer false positives in the placebo tests? 
   

- A: 0.45

- B: 0.05

- C: 0.65

- D: 0.25

In [6]:
num = 1000
real_event_time = num // 2
days = np.arange(num)

np.random.seed(0)

R_market = np.random.normal(0, 1, num) + days / num
R_target = 2 + R_market + make_error(0.9, num) + (days == real_event_time + 1) * 2

placebo_times = [placebo_time for placebo_time in range(3, num - 1) if placebo_time != real_event_time]
placebo_t_values = [get_t_value(R_target, R_market, placebo_time) for placebo_time in placebo_times]

print(np.mean(np.abs(placebo_t_values) > 1.96))

0.04723618090452261


Answer:     B
_____